# Venusian PCW Detector · 科学图表

这是项目的出图入口。安装仓库的 `notebook` 可选依赖，选用对应的 Python 内核，再从上到下运行。

- 默认用仓库中的参考输入重算统计，并生成 9 张图。
- 指定 1 Hz MAG BAM 目录后，可再绘制 3 张事件波形和轨道图。
- 每组图独立保存为 PNG 和 PDF，统一写入 `build/figures/`。可以只重跑需要的图组。

In [ ]:
# 可修改的运行参数
MAG_DIR = None  # 例如 "/path/to/VEX_1Hz"；留空时读取 VENUS_EXPRESS_RAW_DATA_DIR
DRAW_MAG_FIGURES = True  # 有 MAG 目录时绘制波形和轨道图
USE_NEW_SEARCH = False  # True 需要先完成 mission、holdout 搜索
RECOMPUTE_DENSITY_PSD = False  # True 从 MAG 重新测量密度反演所用的波谱
SHOW_INLINE = True

# 按优先顺序列出字体家族；留空使用随附字体和系统字体。
FONT_FAMILIES = []  # 例如 ["DejaVu Sans", "Noto Sans CJK SC"]
FONT_PATHS = []  # 例如 ["/path/to/fonts"]

## 准备统计和出图环境

先校验输入、生成统计表，再从下方选择需要的图组。

In [ ]:
import os
from pathlib import Path

from venus_express.repro.settings import BUILD, FIGURES, display_path

os.environ.setdefault("MPLCONFIGDIR", str(BUILD / "matplotlib"))
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
if MAG_DIR is not None:
    os.environ["VENUS_EXPRESS_RAW_DATA_DIR"] = str(Path(MAG_DIR).expanduser().resolve())
has_mag = bool(os.environ.get("VENUS_EXPRESS_RAW_DATA_DIR"))

from IPython.display import Image, display
from venusian_pcw import figures
from venusian_pcw.fonts import plot_families
from venusian_pcw.statistics import compute

font_stack = plot_families(FONT_FAMILIES, FONT_PATHS)
compute(raw_search=USE_NEW_SEARCH, raw_density=RECOMPUTE_DENSITY_PSD)

generated = []

def show_figures(*names):
    for name in names:
        path = FIGURES / f"{name}.png"
        generated.append(path)
        if SHOW_INLINE:
            display(Image(filename=str(path), width=850))

## 模型结构与评价

In [ ]:
with figures.figure_style(font_stack):
    figures.architecture()
    figures.model_results()
show_figures("02_model_architecture", "03_model_evaluation")

## 年际变化、空间分布与氢密度

In [ ]:
with figures.figure_style(font_stack):
    figures.annual_response()
    figures.spatial_maps()
    figures.density()
show_figures(
    "05_annual_response",
    "11_rx_cluster_occurrence",
    "12_rx_manual_energy",
    "06_h_density",
)

## 太阳活动与弓激波

弓激波面板从 `inputs/boundaries/` 下的数值表绘制。

In [ ]:
with figures.figure_style(font_stack):
    figures.solar_cycle()
    figures.shock_figures()
show_figures("07_solar_cycle", "08_shock_compression", "09_shock_geometry")

## 原始 MAG 观测示例

设置 `MAG_DIR` 或环境变量后，运行这一节生成波形、波谱和事件轨道图。

In [ ]:
if DRAW_MAG_FIGURES and has_mag:
    with figures.figure_style(font_stack):
        figures.wave_event()
        figures.radial_event()
        figures.rx_examples()
    show_figures("04_pcw_event", "10_radial_imf_event", "13_rx_case_orbits")
else:
    print("已跳过 3 张原始 MAG 示例图。")

In [ ]:
unique_figures = list(dict.fromkeys(generated))
print(f"本次生成 {len(unique_figures)} 张图。目录：{display_path(FIGURES)}")
for path in unique_figures:
    print(f"{path.stem}.png / {path.stem}.pdf")